# Elasticsearch 核心概念

## 从关系型数据库理解 Elasticsearch

可以先用一个不完全严格、但便于入门的类比：

| 关系型数据库 | Elasticsearch |
|---|---|
| Database | Cluster 中的一组索引 |
| Table | Index |
| Row | Document |
| Column | Field |
| Schema | Mapping |
| SQL | Query DSL |

例如本 Notebook 使用：

```text
Index: blog_posts_py

Document:
{
    "title": "Elasticsearch 入门指南",
    "content": "...",
    "tags": ["Elasticsearch", "教程", "搜索"],
    "author": "张三",
    "created_at": "2023-10-26T10:00:00"
}
```

Elasticsearch 的核心不是“把 JSON 存进去”这么简单，而是：

```text
原始文本
↓
Analyzer 分词
↓
Token
↓
倒排索引
↓
Query
↓
相关性计算
↓
Top K 文档
```

## Index

Index 是一组结构相似的文档集合。

例如：

```text
blog_posts_py
```

可以理解成“博客文章表”。

## Document

Document 是 Elasticsearch 中最基本的数据单元，通常是一个 JSON 对象。

## Mapping

Mapping 定义字段类型和分析方式，例如：

```json
"title": {
  "type": "text",
  "analyzer": "ik_max_word",
  "search_analyzer": "ik_smart"
}
```

它告诉 Elasticsearch：

- `title` 是全文检索字段
- 写入时使用 `ik_max_word`
- 查询时使用 `ik_smart`

## text 与 keyword

这是 Elasticsearch 入门中最重要的区别之一。

`text`：

- 会被 Analyzer 分词
- 用于全文检索
- 常配合 `match`
- 例如标题、正文

`keyword`：

- 整体作为一个值
- 不进行全文分词
- 用于精确匹配、过滤、排序、聚合
- 例如作者、状态、标签、类别

因此：

```text
title   → text
content → text
author  → keyword
tags    → keyword
```


# 倒排索引与全文检索

## 什么是倒排索引

假设有三篇文章：

```text
doc1: Elasticsearch 搜索教程
doc2: Elasticsearch 中文分词
doc3: Python 搜索实践
```

经过分词后，可以形成类似：

```text
Elasticsearch → doc1, doc2
搜索          → doc1, doc3
Python        → doc3
教程          → doc1
```

这就是倒排索引的核心思想：

> 不从“文档找词”，而是从“词找文档”。

因此搜索 `"Elasticsearch"` 时，不需要逐条扫描所有文章，而可以快速找到包含对应 Term 的文档。

## Index Time 与 Search Time

全文检索会发生两次文本分析：

```text
写入文档
↓
Index Analyzer
↓
建立倒排索引

用户输入 Query
↓
Search Analyzer
↓
生成查询 Term
↓
匹配倒排索引
```

在这个 Notebook 中：

```text
Index Analyzer  = ik_max_word
Search Analyzer = ik_smart
```

`ik_max_word` 倾向于更细粒度切分，有利于扩大召回；`ik_smart` 倾向于较粗粒度切分，适合作为较简洁的查询分析方式。

注意：IK 并不是 Elasticsearch 内置 Analyzer，需要提前安装与 Elasticsearch 版本匹配的 IK Analysis 插件。


# 环境准备

## Python Client 版本

建议让 Python Client 的主版本与 Elasticsearch Server 主版本保持匹配。

例如：

```text
Elasticsearch 8.x → elasticsearch Python client 8.x
Elasticsearch 9.x → elasticsearch Python client 9.x
```

如果你的环境已经安装，可以跳过下面的安装单元。


In [1]:
# 在 Jupyter 中安装 Elasticsearch Python Client
# 如果已经安装，可跳过。
%pip install -q elasticsearch


Note: you may need to restart the kernel to use updated packages.


## 连接 Elasticsearch

下面默认连接：

```text
http://localhost:9200
```

如果你的 Elasticsearch 开启了默认安全认证，通常需要使用 HTTPS，并提供账号密码、API Key 或 CA 证书。

本实验为了突出 Elasticsearch 查询知识，先以“本地无认证环境”为例。


In [2]:
from elasticsearch import Elasticsearch
import elasticsearch

ELASTICSEARCH_URL = "http://localhost:9200"

# 无认证本地环境
es_client = Elasticsearch(ELASTICSEARCH_URL)

print("Python Client 版本:", elasticsearch.__version__)

if es_client.ping():
    print("连接成功！")
    info = es_client.info()
    print("Elasticsearch Server 版本:", info["version"]["number"])
else:
    raise RuntimeError("连接失败，请检查 Elasticsearch 是否启动、地址是否正确以及认证配置。")


/opt/miniconda3/envs/py312/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.3) doesn't match a supported version!
  warnings.warn(


Python Client 版本: (9, 1, 0)
连接成功！
Elasticsearch Server 版本: 9.1.3


## 如果开启了安全认证

以下代码只是配置示例，不要同时执行所有方案。

```python
# 用户名 + 密码
es_client = Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic", "你的密码"),
    verify_certs=False
)

# API Key
es_client = Elasticsearch(
    "https://localhost:9200",
    api_key="你的_api_key"
)

# 使用 CA 证书
es_client = Elasticsearch(
    "https://localhost:9200",
    ca_certs="/path/to/http_ca.crt",
    basic_auth=("elastic", "你的密码")
)
```

生产环境不要为了方便长期关闭证书校验。


# IK 中文分词器检查

本 Notebook 的 Mapping 使用：

```text
ik_max_word
ik_smart
```

如果 IK 插件未安装，创建索引时通常会看到类似：

```text
Unknown analyzer [ik_max_word]
```

可以先查看节点插件。


In [3]:
# 查看 Elasticsearch 节点上安装的插件
plugins = es_client.cat.plugins(format="json")

if plugins:
    for item in plugins:
        print(item)
else:
    print("当前没有返回插件信息。若后续出现 Unknown analyzer，请检查 IK 插件是否安装。")


{'name': 'lyzdeMacBook-Pro.local', 'component': 'analysis-ik', 'version': '9.1.3'}


# 创建索引与 Mapping

我们创建一个博客索引：

```text
blog_posts_py
```

字段设计：

| 字段 | 类型 | 作用 |
|---|---|---|
| title | text | 标题全文检索 |
| content | text | 正文全文检索 |
| tags | keyword | 标签精确过滤与聚合 |
| author | keyword | 作者精确过滤与聚合 |
| created_at | date | 日期范围查询与排序 |

## 为什么 title/content 用 text

因为：

```text
"Elasticsearch 入门指南"
```

需要被拆成多个可搜索 Term。

## 为什么 author/tags 用 keyword

例如：

```text
author = "张三"
```

通常希望执行：

```text
张三 == 张三
```

而不是把作者名继续做全文检索。


In [4]:
index_name = "blog_posts_py"

settings = {
    "number_of_shards": 1,
    "number_of_replicas": 0
}

mappings = {
    "properties": {
        "title": {
            "type": "text",
            "analyzer": "ik_max_word",
            "search_analyzer": "ik_smart"
        },
        "content": {
            "type": "text",
            "analyzer": "ik_max_word",
            "search_analyzer": "ik_smart"
        },
        "tags": {
            "type": "keyword"
        },
        "author": {
            "type": "keyword"
        },
        "created_at": {
            "type": "date"
        }
    }
}

if not es_client.indices.exists(index=index_name):
    es_client.indices.create(
        index=index_name,
        settings=settings,
        mappings=mappings
    )
    print(f"索引 '{index_name}' 创建成功。")
else:
    print(f"索引 '{index_name}' 已存在。")


索引 'blog_posts_py' 创建成功。


## 查看 Mapping

Mapping 一旦创建后，某些字段核心类型和 Analyzer 不能随意原地修改。

教学实验中，如果 Mapping 设计错了，最简单的处理通常是：

```text
删除旧索引
↓
重新创建索引
↓
重新导入数据
```

生产环境则通常采用新索引 + Reindex + Alias 切换。


In [5]:
mapping_info = es_client.indices.get_mapping(index=index_name)
mapping_info


ObjectApiResponse({'blog_posts_py': {'mappings': {'properties': {'author': {'type': 'keyword'}, 'content': {'type': 'text', 'analyzer': 'ik_max_word', 'search_analyzer': 'ik_smart'}, 'created_at': {'type': 'date'}, 'tags': {'type': 'keyword'}, 'title': {'type': 'text', 'analyzer': 'ik_max_word', 'search_analyzer': 'ik_smart'}}}}})

# 分词实验

在真正搜索之前，先理解 Analyzer 对文本做了什么。

使用 `_analyze` API 可以直接观察 Token。

例如：

```text
"Elasticsearch 中文搜索技术"
```

分别使用 `ik_max_word` 和 `ik_smart`。


In [6]:
sample_text = "Elasticsearch 中文搜索技术"

for analyzer in ["ik_max_word", "ik_smart"]:
    result = es_client.indices.analyze(
        index=index_name,
        analyzer=analyzer,
        text=sample_text
    )
    tokens = [item["token"] for item in result["tokens"]]
    print(f"{analyzer}:")
    print(tokens)
    print()


ik_max_word:
['elasticsearch', '中文搜索', '中文', '搜索', '技术']

ik_smart:
['elasticsearch', '中文搜索', '技术']



# 写入实验数据

原始示例只有两篇文档，不利于观察查询差异。

这里扩展成 8 篇博客数据，覆盖：

- Elasticsearch
- IK 中文分词
- Python Client
- RAG
- 向量检索
- BM25
- MySQL
- Redis

为了让 Notebook 可以重复执行，我们给每篇文档指定固定 `_id`。


In [7]:
from datetime import datetime
from elasticsearch import helpers

documents = [
    {
        "_id": "1",
        "title": "Elasticsearch 入门指南",
        "content": "这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。",
        "tags": ["Elasticsearch", "教程", "搜索"],
        "author": "张三",
        "created_at": datetime(2023, 10, 26, 10, 0, 0)
    },
    {
        "_id": "2",
        "title": "深入理解 IK 分词器",
        "content": "IK 分词器是中文分词的优秀工具。它的智能分词和细粒度分词模式各有优势。",
        "tags": ["分词", "IK", "中文"],
        "author": "李四",
        "created_at": datetime(2023, 10, 25, 15, 30, 0)
    },
    {
        "_id": "3",
        "title": "Python 操作 Elasticsearch",
        "content": "使用 Elasticsearch Python Client 可以创建索引、写入文档并执行全文搜索。",
        "tags": ["Elasticsearch", "Python", "教程"],
        "author": "张三",
        "created_at": datetime(2023, 11, 1, 9, 0, 0)
    },
    {
        "_id": "4",
        "title": "RAG 中的混合检索",
        "content": "RAG 系统可以组合 BM25 关键词检索和向量检索，提高知识库搜索的召回能力。",
        "tags": ["RAG", "搜索", "向量检索"],
        "author": "王五",
        "created_at": datetime(2024, 1, 12, 14, 20, 0)
    },
    {
        "_id": "5",
        "title": "BM25 相关性排序原理",
        "content": "BM25 会考虑词频、逆文档频率和文档长度，是 Elasticsearch 全文搜索中的经典相关性算法。",
        "tags": ["Elasticsearch", "BM25", "搜索"],
        "author": "王五",
        "created_at": datetime(2024, 2, 8, 11, 10, 0)
    },
    {
        "_id": "6",
        "title": "向量数据库与语义搜索",
        "content": "Embedding 将文本转换成向量，向量检索可以找到没有出现相同关键词但语义相关的内容。",
        "tags": ["Embedding", "向量检索", "搜索"],
        "author": "赵六",
        "created_at": datetime(2024, 3, 5, 16, 0, 0)
    },
    {
        "_id": "7",
        "title": "MySQL 索引优化实践",
        "content": "本文介绍 B+Tree、联合索引和慢查询优化方法，重点讨论关系型数据库查询性能。",
        "tags": ["MySQL", "数据库", "性能"],
        "author": "李四",
        "created_at": datetime(2024, 4, 2, 10, 30, 0)
    },
    {
        "_id": "8",
        "title": "Redis 缓存设计",
        "content": "Redis 常用于缓存热点数据，需要关注缓存穿透、缓存击穿和缓存雪崩问题。",
        "tags": ["Redis", "缓存", "数据库"],
        "author": "赵六",
        "created_at": datetime(2024, 5, 18, 13, 40, 0)
    }
]

actions = [
    {
        "_index": index_name,
        "_id": doc["_id"],
        "_source": {k: v for k, v in doc.items() if k != "_id"}
    }
    for doc in documents
]

success_count, errors = helpers.bulk(
    es_client,
    actions,
    raise_on_error=False
)

es_client.indices.refresh(index=index_name)

print("成功写入:", success_count)
print("错误数量:", len(errors))


成功写入: 8
错误数量: 0


# 搜索结果辅助函数

后续所有查询都复用这个函数。

重点观察：

```text
_index
_id
_score
_source
```

其中 `_score` 表示相关性得分。默认全文检索通常会按照 `_score` 从高到低排序。


In [8]:
def search_docs(query, size=10, sort=None, source=None, highlight=None):
    kwargs = {
        "index": index_name,
        "query": query,
        "size": size
    }

    if sort is not None:
        kwargs["sort"] = sort

    if source is not None:
        kwargs["source"] = source

    if highlight is not None:
        kwargs["highlight"] = highlight

    response = es_client.search(**kwargs)

    total = response["hits"]["total"]["value"]
    print(f"找到 {total} 条文档：")

    for hit in response["hits"]["hits"]:
        source_doc = hit["_source"]
        print(
            f"id={hit['_id']} | "
            f"score={hit.get('_score')} | "
            f"title={source_doc.get('title')} | "
            f"author={source_doc.get('author')}"
        )

        if "highlight" in hit:
            print("highlight:", hit["highlight"])

    return response


# match：全文检索

`match` 是最常用的全文搜索 Query。

例如：

```json
{
  "match": {
    "title": "入门指南"
  }
}
```

执行逻辑不是简单的字符串包含判断，而更接近：

```text
Query Text
↓
使用字段的 Search Analyzer 分词
↓
得到 Query Terms
↓
查询倒排索引
↓
计算相关性得分
↓
返回结果
```

因此 `match` 特别适合 `text` 字段。


In [9]:
query_match = {
    "match": {
        "title": "入门指南"
    }
}

search_docs(query_match)


找到 1 条文档：
id=1 | score=4.284642 | title=Elasticsearch 入门指南 | author=张三


ObjectApiResponse({'took': 21, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 4.284642, 'hits': [{'_index': 'blog_posts_py', '_id': '1', '_score': 4.284642, '_source': {'title': 'Elasticsearch 入门指南', 'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。', 'tags': ['Elasticsearch', '教程', '搜索'], 'author': '张三', 'created_at': '2023-10-26T10:00:00'}}]}})

# term：精确匹配

`term` 不会像 `match` 那样先对查询文本做全文分析，通常用于精确值匹配。

例如作者字段：

```text
author = keyword
```

所以非常适合：

```json
{
  "term": {
    "author": "张三"
  }
}
```

可以记成：

```text
match → 面向全文检索
term  → 面向精确 Term
```

一个常见错误是对 `text` 字段随意使用 `term`，然后发现“明明看起来包含这个字符串，却搜不到”。

原因往往是：索引中实际保存的是 Analyzer 处理后的 Term。


In [10]:
query_term = {
    "term": {
        "author": "张三"
    }
}

search_docs(query_term)


找到 2 条文档：
id=1 | score=1.2809337 | title=Elasticsearch 入门指南 | author=张三
id=3 | score=1.2809337 | title=Python 操作 Elasticsearch | author=张三


ObjectApiResponse({'took': 3, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 1.2809337, 'hits': [{'_index': 'blog_posts_py', '_id': '1', '_score': 1.2809337, '_source': {'title': 'Elasticsearch 入门指南', 'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。', 'tags': ['Elasticsearch', '教程', '搜索'], 'author': '张三', 'created_at': '2023-10-26T10:00:00'}}, {'_index': 'blog_posts_py', '_id': '3', '_score': 1.2809337, '_source': {'title': 'Python 操作 Elasticsearch', 'content': '使用 Elasticsearch Python Client 可以创建索引、写入文档并执行全文搜索。', 'tags': ['Elasticsearch', 'Python', '教程'], 'author': '张三', 'created_at': '2023-11-01T09:00:00'}}]}})

# bool：组合多个检索条件

`bool` 是 Elasticsearch Query DSL 中最核心的组合能力。

常见四种子句：

| 子句 | 含义 | 通常影响 `_score` |
|---|---|---|
| must | 必须满足 | 是 |
| filter | 必须满足，但主要用于过滤 | 否 |
| should | 最好满足，可用于提升得分 | 是 |
| must_not | 必须不满足 | 否 |

例如：

```text
正文必须与“搜索技术”相关
AND
作者必须是“张三”
```

可以写成：

```text
must:
    match(content, "搜索技术")

filter:
    term(author, "张三")
```

这里作者只是一个硬条件，没有必要参与相关性计算，因此适合放 `filter`。


In [11]:
query_bool = {
    "bool": {
        "must": [
            {
                "match": {
                    "content": "搜索技术"
                }
            }
        ],
        "filter": [
            {
                "term": {
                    "author": "张三"
                }
            }
        ]
    }
}

search_docs(query_bool)


找到 2 条文档：
id=1 | score=2.4126868 | title=Elasticsearch 入门指南 | author=张三
id=3 | score=0.81514835 | title=Python 操作 Elasticsearch | author=张三


ObjectApiResponse({'took': 6, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 2.4126868, 'hits': [{'_index': 'blog_posts_py', '_id': '1', '_score': 2.4126868, '_source': {'title': 'Elasticsearch 入门指南', 'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。', 'tags': ['Elasticsearch', '教程', '搜索'], 'author': '张三', 'created_at': '2023-10-26T10:00:00'}}, {'_index': 'blog_posts_py', '_id': '3', '_score': 0.81514835, '_source': {'title': 'Python 操作 Elasticsearch', 'content': '使用 Elasticsearch Python Client 可以创建索引、写入文档并执行全文搜索。', 'tags': ['Elasticsearch', 'Python', '教程'], 'author': '张三', 'created_at': '2023-11-01T09:00:00'}}]}})

# Query 与 Filter 的区别

可以把搜索条件分成两类：

```text
Query:
“哪篇文章和我的问题更相关？”

Filter:
“哪些文章有资格进入候选集合？”
```

例如：

```text
搜索主题 = Elasticsearch
作者 = 张三
日期 >= 2023-10-01
```

其中：

```text
Elasticsearch → Query
张三          → Filter
日期范围      → Filter
```

这是设计复杂检索时非常重要的思维方式。


# multi_match：多个字段一起搜索

真实搜索系统通常不会只查标题或者只查正文。

例如用户搜索：

```text
Elasticsearch 搜索
```

可能希望同时搜索：

```text
title
content
```

可以使用 `multi_match`。

还可以通过：

```text
title^2
```

表示标题的重要性更高。


In [12]:
query_multi_match = {
    "multi_match": {
        "query": "Elasticsearch 搜索",
        "fields": [
            "title^2",
            "content"
        ]
    }
}

search_docs(query_multi_match)


找到 5 条文档：
id=6 | score=3.0795865 | title=向量数据库与语义搜索 | author=赵六
id=1 | score=3.0631027 | title=Elasticsearch 入门指南 | author=张三
id=3 | score=3.0631027 | title=Python 操作 Elasticsearch | author=张三
id=5 | score=1.5597922 | title=BM25 相关性排序原理 | author=王五
id=4 | score=0.6602099 | title=RAG 中的混合检索 | author=王五


ObjectApiResponse({'took': 23, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 3.0795865, 'hits': [{'_index': 'blog_posts_py', '_id': '6', '_score': 3.0795865, '_source': {'title': '向量数据库与语义搜索', 'content': 'Embedding 将文本转换成向量，向量检索可以找到没有出现相同关键词但语义相关的内容。', 'tags': ['Embedding', '向量检索', '搜索'], 'author': '赵六', 'created_at': '2024-03-05T16:00:00'}}, {'_index': 'blog_posts_py', '_id': '1', '_score': 3.0631027, '_source': {'title': 'Elasticsearch 入门指南', 'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。', 'tags': ['Elasticsearch', '教程', '搜索'], 'author': '张三', 'created_at': '2023-10-26T10:00:00'}}, {'_index': 'blog_posts_py', '_id': '3', '_score': 3.0631027, '_source': {'title': 'Python 操作 Elasticsearch', 'content': '使用 Elasticsearch Python Client 可以创建索引、写入文档并执行全文搜索。', 'tags': ['Elasticsearch', 'Python', '教程'], 'author': '张三', 'created_at': '2023-11-01T09:00:00'}}, {'_index': 'b

# match_phrase：短语匹配

普通 `match` 更关注 Term 是否匹配。

如果希望单词/Token 之间保持更强的相邻与顺序关系，可以使用 `match_phrase`。

例如：

```text
向量检索
```

相比拆开理解“向量”和“检索”，Phrase Query 更强调短语结构。


In [13]:
query_phrase = {
    "match_phrase": {
        "content": "向量检索"
    }
}

search_docs(query_phrase)


找到 2 条文档：
id=6 | score=2.4874108 | title=向量数据库与语义搜索 | author=赵六
id=4 | score=2.4401317 | title=RAG 中的混合检索 | author=王五


ObjectApiResponse({'took': 23, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 2.4874108, 'hits': [{'_index': 'blog_posts_py', '_id': '6', '_score': 2.4874108, '_source': {'title': '向量数据库与语义搜索', 'content': 'Embedding 将文本转换成向量，向量检索可以找到没有出现相同关键词但语义相关的内容。', 'tags': ['Embedding', '向量检索', '搜索'], 'author': '赵六', 'created_at': '2024-03-05T16:00:00'}}, {'_index': 'blog_posts_py', '_id': '4', '_score': 2.4401317, '_source': {'title': 'RAG 中的混合检索', 'content': 'RAG 系统可以组合 BM25 关键词检索和向量检索，提高知识库搜索的召回能力。', 'tags': ['RAG', '搜索', '向量检索'], 'author': '王五', 'created_at': '2024-01-12T14:20:00'}}]}})

# terms：多个精确值

`term` 是单个精确值：

```text
author = 张三
```

`terms` 是多个精确值中的任意一个：

```text
tags in ["搜索", "RAG"]
```

特别适合 `keyword`、ID、状态、标签等字段。


In [14]:
query_terms = {
    "terms": {
        "tags": ["搜索", "RAG"]
    }
}

search_docs(query_terms)


找到 4 条文档：
id=1 | score=1.0 | title=Elasticsearch 入门指南 | author=张三
id=4 | score=1.0 | title=RAG 中的混合检索 | author=王五
id=5 | score=1.0 | title=BM25 相关性排序原理 | author=王五
id=6 | score=1.0 | title=向量数据库与语义搜索 | author=赵六


ObjectApiResponse({'took': 67, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 4, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'blog_posts_py', '_id': '1', '_score': 1.0, '_source': {'title': 'Elasticsearch 入门指南', 'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。', 'tags': ['Elasticsearch', '教程', '搜索'], 'author': '张三', 'created_at': '2023-10-26T10:00:00'}}, {'_index': 'blog_posts_py', '_id': '4', '_score': 1.0, '_source': {'title': 'RAG 中的混合检索', 'content': 'RAG 系统可以组合 BM25 关键词检索和向量检索，提高知识库搜索的召回能力。', 'tags': ['RAG', '搜索', '向量检索'], 'author': '王五', 'created_at': '2024-01-12T14:20:00'}}, {'_index': 'blog_posts_py', '_id': '5', '_score': 1.0, '_source': {'title': 'BM25 相关性排序原理', 'content': 'BM25 会考虑词频、逆文档频率和文档长度，是 Elasticsearch 全文搜索中的经典相关性算法。', 'tags': ['Elasticsearch', 'BM25', '搜索'], 'author': '王五', 'created_at': '2024-02-08T11:10:00'}}, {'_index': 'blog_posts_py', '_id': '6', '_score': 1.0, '_so

# range：范围查询

日期、价格、年龄、分数等字段经常需要范围过滤。

常用操作符：

```text
gt  > 
gte >=
lt  <
lte <=
```

例如查询：

```text
created_at >= 2024-01-01
```


In [15]:
query_range = {
    "range": {
        "created_at": {
            "gte": "2024-01-01"
        }
    }
}

search_docs(query_range)


找到 5 条文档：
id=4 | score=1.0 | title=RAG 中的混合检索 | author=王五
id=5 | score=1.0 | title=BM25 相关性排序原理 | author=王五
id=6 | score=1.0 | title=向量数据库与语义搜索 | author=赵六
id=7 | score=1.0 | title=MySQL 索引优化实践 | author=李四
id=8 | score=1.0 | title=Redis 缓存设计 | author=赵六


ObjectApiResponse({'took': 37, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 5, 'relation': 'eq'}, 'max_score': 1.0, 'hits': [{'_index': 'blog_posts_py', '_id': '4', '_score': 1.0, '_source': {'title': 'RAG 中的混合检索', 'content': 'RAG 系统可以组合 BM25 关键词检索和向量检索，提高知识库搜索的召回能力。', 'tags': ['RAG', '搜索', '向量检索'], 'author': '王五', 'created_at': '2024-01-12T14:20:00'}}, {'_index': 'blog_posts_py', '_id': '5', '_score': 1.0, '_source': {'title': 'BM25 相关性排序原理', 'content': 'BM25 会考虑词频、逆文档频率和文档长度，是 Elasticsearch 全文搜索中的经典相关性算法。', 'tags': ['Elasticsearch', 'BM25', '搜索'], 'author': '王五', 'created_at': '2024-02-08T11:10:00'}}, {'_index': 'blog_posts_py', '_id': '6', '_score': 1.0, '_source': {'title': '向量数据库与语义搜索', 'content': 'Embedding 将文本转换成向量，向量检索可以找到没有出现相同关键词但语义相关的内容。', 'tags': ['Embedding', '向量检索', '搜索'], 'author': '赵六', 'created_at': '2024-03-05T16:00:00'}}, {'_index': 'blog_posts_py', '_id': '7', '_score': 1.0, '_source': {'title':

# should：可选条件与相关性提升

假设用户搜索：

```text
搜索
```

但我们更希望带有：

```text
Elasticsearch
```

标签的文章排在前面。

可以使用：

```text
must   → 内容必须与“搜索”相关
should → 标签最好是“Elasticsearch”
```

`should` 可以用于表达“加分项”。


In [16]:
query_should = {
    "bool": {
        "must": [
            {
                "match": {
                    "content": "搜索"
                }
            }
        ],
        "should": [
            {
                "term": {
                    "tags": {
                        "value": "Elasticsearch",
                        "boost": 2.0
                    }
                }
            }
        ]
    }
}

search_docs(query_should)


找到 4 条文档：
id=3 | score=3.4124177 | title=Python 操作 Elasticsearch | author=张三
id=1 | score=3.2702713 | title=Elasticsearch 入门指南 | author=张三
id=5 | score=3.2574792 | title=BM25 相关性排序原理 | author=王五
id=4 | score=0.6602099 | title=RAG 中的混合检索 | author=王五


ObjectApiResponse({'took': 6, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 4, 'relation': 'eq'}, 'max_score': 3.4124177, 'hits': [{'_index': 'blog_posts_py', '_id': '3', '_score': 3.4124177, '_source': {'title': 'Python 操作 Elasticsearch', 'content': '使用 Elasticsearch Python Client 可以创建索引、写入文档并执行全文搜索。', 'tags': ['Elasticsearch', 'Python', '教程'], 'author': '张三', 'created_at': '2023-11-01T09:00:00'}}, {'_index': 'blog_posts_py', '_id': '1', '_score': 3.2702713, '_source': {'title': 'Elasticsearch 入门指南', 'content': '这是一篇关于如何安装和使用 Elasticsearch 的详细文章。学习搜索技术可以提升数据处理能力。', 'tags': ['Elasticsearch', '教程', '搜索'], 'author': '张三', 'created_at': '2023-10-26T10:00:00'}}, {'_index': 'blog_posts_py', '_id': '5', '_score': 3.2574792, '_source': {'title': 'BM25 相关性排序原理', 'content': 'BM25 会考虑词频、逆文档频率和文档长度，是 Elasticsearch 全文搜索中的经典相关性算法。', 'tags': ['Elasticsearch', 'BM25', '搜索'], 'author': '王五', 'created_at': '2024-02-08T11:10:00'}}, {

# must_not：排除条件

例如：

```text
搜索所有与数据库相关的文章
但排除 Redis
```

可以将排除规则放进 `must_not`。


In [17]:
query_must_not = {
    "bool": {
        "must": [
            {
                "match": {
                    "content": "数据库"
                }
            }
        ],
        "must_not": [
            {
                "term": {
                    "tags": "Redis"
                }
            }
        ]
    }
}

search_docs(query_must_not)


找到 1 条文档：
id=7 | score=1.7740581 | title=MySQL 索引优化实践 | author=李四


ObjectApiResponse({'took': 8, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 1, 'relation': 'eq'}, 'max_score': 1.7740581, 'hits': [{'_index': 'blog_posts_py', '_id': '7', '_score': 1.7740581, '_source': {'title': 'MySQL 索引优化实践', 'content': '本文介绍 B+Tree、联合索引和慢查询优化方法，重点讨论关系型数据库查询性能。', 'tags': ['MySQL', '数据库', '性能'], 'author': '李四', 'created_at': '2024-04-02T10:30:00'}}]}})

# 分页与排序

## 分页

Elasticsearch 常见分页参数：

```text
from
size
```

例如：

```text
from = 0
size = 3
```

表示第一页取 3 条。

但深分页时不建议无限增大 `from`，大规模线上检索更常考虑 `search_after`。

## 排序

全文检索默认通常更关注：

```text
_score
```

业务场景也经常按：

```text
created_at
price
sales
```

等字段排序。


In [18]:
response = es_client.search(
    index=index_name,
    query={"match_all": {}},
    from_=0,
    size=3,
    sort=[
        {"created_at": {"order": "desc"}}
    ]
)

for hit in response["hits"]["hits"]:
    print(hit["_source"]["created_at"], hit["_source"]["title"])


2024-05-18T13:40:00 Redis 缓存设计
2024-04-02T10:30:00 MySQL 索引优化实践
2024-03-05T16:00:00 向量数据库与语义搜索


# 高亮搜索结果

搜索页面经常需要将命中的关键词突出显示。

Elasticsearch 可以在 Search API 中配置 `highlight`。

例如搜索：

```text
向量检索
```

并高亮正文命中的位置。


In [19]:
query_highlight = {
    "match": {
        "content": "向量检索"
    }
}

highlight = {
    "fields": {
        "content": {}
    },
    "pre_tags": ["<mark>"],
    "post_tags": ["</mark>"]
}

search_docs(
    query_highlight,
    highlight=highlight
)


找到 2 条文档：
id=6 | score=2.9694743 | title=向量数据库与语义搜索 | author=赵六
highlight: {'content': ['Embedding 将文本转换成<mark>向量</mark>，<mark>向量</mark><mark>检索</mark>可以找到没有出现相同关键词但语义相关的内容。']}
id=4 | score=2.922943 | title=RAG 中的混合检索 | author=王五
highlight: {'content': ['RAG 系统可以组合 BM25 关键词<mark>检索</mark>和<mark>向量</mark><mark>检索</mark>，提高知识库搜索的召回能力。']}


ObjectApiResponse({'took': 67, 'timed_out': False, '_shards': {'total': 1, 'successful': 1, 'skipped': 0, 'failed': 0}, 'hits': {'total': {'value': 2, 'relation': 'eq'}, 'max_score': 2.9694743, 'hits': [{'_index': 'blog_posts_py', '_id': '6', '_score': 2.9694743, '_source': {'title': '向量数据库与语义搜索', 'content': 'Embedding 将文本转换成向量，向量检索可以找到没有出现相同关键词但语义相关的内容。', 'tags': ['Embedding', '向量检索', '搜索'], 'author': '赵六', 'created_at': '2024-03-05T16:00:00'}, 'highlight': {'content': ['Embedding 将文本转换成<mark>向量</mark>，<mark>向量</mark><mark>检索</mark>可以找到没有出现相同关键词但语义相关的内容。']}}, {'_index': 'blog_posts_py', '_id': '4', '_score': 2.922943, '_source': {'title': 'RAG 中的混合检索', 'content': 'RAG 系统可以组合 BM25 关键词检索和向量检索，提高知识库搜索的召回能力。', 'tags': ['RAG', '搜索', '向量检索'], 'author': '王五', 'created_at': '2024-01-12T14:20:00'}, 'highlight': {'content': ['RAG 系统可以组合 BM25 关键词<mark>检索</mark>和<mark>向量</mark><mark>检索</mark>，提高知识库搜索的召回能力。']}}]}})

# Aggregation：聚合分析

Elasticsearch 不只是搜索引擎，也能够做聚合分析。

例如统计：

```text
每个作者有多少篇文章
```

作者字段是 `keyword`，非常适合 `terms aggregation`。

可以理解为类似 SQL：

```sql
SELECT author, COUNT(*)
FROM blog_posts
GROUP BY author;
```


In [20]:
response = es_client.search(
    index=index_name,
    size=0,
    aggs={
        "articles_by_author": {
            "terms": {
                "field": "author",
                "size": 10
            }
        }
    }
)

buckets = response["aggregations"]["articles_by_author"]["buckets"]

for bucket in buckets:
    print(bucket["key"], bucket["doc_count"])


张三 2
李四 2
王五 2
赵六 2


## 标签聚合

`tags` 是 keyword 数组。

同一篇文档可以同时属于多个标签，因此可以直接统计不同标签出现的文档数量。


In [21]:
response = es_client.search(
    index=index_name,
    size=0,
    aggs={
        "popular_tags": {
            "terms": {
                "field": "tags",
                "size": 20
            }
        }
    }
)

for bucket in response["aggregations"]["popular_tags"]["buckets"]:
    print(bucket["key"], bucket["doc_count"])


搜索 4
Elasticsearch 3
向量检索 2
教程 2
数据库 2
BM25 1
Embedding 1
IK 1
MySQL 1
Python 1
RAG 1
Redis 1
中文 1
分词 1
性能 1
缓存 1


# CRUD 基础操作

除了 Search，还应掌握最基本的：

```text
Create / Index
Get
Update
Delete
```

下面用一个临时文档演示，最后再删除，不影响主要实验数据。


In [22]:
temp_id = "temp-1"

# 写入
es_client.index(
    index=index_name,
    id=temp_id,
    document={
        "title": "临时测试文档",
        "content": "这是一条用于 CRUD 演示的临时文档。",
        "tags": ["测试"],
        "author": "测试用户",
        "created_at": datetime.now()
    },
    refresh=True
)

# 获取
doc = es_client.get(index=index_name, id=temp_id)
print("GET:", doc["_source"])

# 更新
es_client.update(
    index=index_name,
    id=temp_id,
    doc={
        "author": "已更新用户"
    },
    refresh=True
)

doc = es_client.get(index=index_name, id=temp_id)
print("UPDATE:", doc["_source"])

# 删除
es_client.delete(
    index=index_name,
    id=temp_id,
    refresh=True
)

print("DELETE 完成")


GET: {'title': '临时测试文档', 'content': '这是一条用于 CRUD 演示的临时文档。', 'tags': ['测试'], 'author': '测试用户', 'created_at': '2026-08-27T21:30:02.823879'}
UPDATE: {'title': '临时测试文档', 'content': '这是一条用于 CRUD 演示的临时文档。', 'tags': ['测试'], 'author': '已更新用户', 'created_at': '2026-08-27T21:30:02.823879'}
DELETE 完成
